In [ ]:
%pip install sentence-transformers pandas numpy tqdm


In [2]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


c:\Users\devia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
books = pd.read_csv('books_enriched.csv')
books.head(3)

,ISBN,Title,Author,Year,Publisher,synopsis,subjects,genres,language,pages,binding,image_url,overview
0,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,"In a small town in Canada, Clara Callan reluct...","Fiction, Historical, Literary, Psychological, ...","Fiction, Historical, Literary",en,414.0,Hardcover,https://images.isbndb.com/covers/1914160348218...,NaN
1,0399135782,The Kitchen God's Wife,Amy Tan,1991,Putnam Pub Group,A Chinese immigrant who is convinced she is dy...,"CALIFORNIA_FICTION, FICTION_FAMILY LIFE_GENERA...","CALIFORNIA_FICTION, FICTION_FAMILY LIFE_GENERA...",en,415.0,Hardcover,https://images.isbndb.com/covers/2267113482330...,NaN
2,0440234743,The Testament,John Grisham,1999,Dell,Heart of darkness... <br>In a plush Virginia o...,"Fiction, Thrillers, Suspense, Legal","Fiction, Thrillers, Suspense",en,533.0,Paperback,https://images.isbndb.com/covers/2005570348234...,NaN


# Book Embeddings

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Helper functions for turning features into descriptive text

Genras and Subjects

In [5]:
def build_genre_subject_text(genres, subjects):
    """
    Convert genres and subjects into a natural language description.
    """
    parts = []
    
    if pd.notna(genres):
        parts.append(f"Genres: {genres}")
        
    if pd.notna(subjects):
        parts.append(f"Subjects: {subjects}")
        
    return ". ".join(parts)


Numerical Features

In [7]:
def build_numerical_text(year, pages):
    """
    Encode numerical features as descriptive text.
    """
    parts = []
    
    if pd.notna(year):
        parts.append(f"Published in the year {int(year)}")
        
    if pd.notna(pages):
        if pages < 200:
            length = "short novel"
        elif pages < 400:
            length = "medium-length novel"
        else:
            length = "long novel"
        parts.append(f"Length: {length}")
        
    return ". ".join(parts)


Feature Weights

In [6]:
WEIGHTS = {
    "synopsis": 0.6,
    "genres_subjects": 0.3,
    "numerical": 0.1
}

Embedding Function

In [8]:
def create_book_embedding(row, model):
    """
    Create a weighted embedding for a single book.
    """
    embeddings = []
    weights = []
    
    # Synopsis embedding
    if pd.notna(row["synopsis"]):
        synopsis_emb = model.encode(row["synopsis"], normalize_embeddings=True)
        embeddings.append(synopsis_emb)
        weights.append(WEIGHTS["synopsis"])
    
    # Genres & Subjects embedding
    gs_text = build_genre_subject_text(row["genres"], row["subjects"])
    if gs_text:
        gs_emb = model.encode(gs_text, normalize_embeddings=True)
        embeddings.append(gs_emb)
        weights.append(WEIGHTS["genres_subjects"])
    
    # Numerical features embedding
    num_text = build_numerical_text(row["Year"], row["pages"])
    if num_text:
        num_emb = model.encode(num_text, normalize_embeddings=True)
        embeddings.append(num_emb)
        weights.append(WEIGHTS["numerical"])
    
    # Weighted sum
    final_embedding = np.zeros(embeddings[0].shape)
    for emb, w in zip(embeddings, weights):
        final_embedding += w * emb
    
    # L2 normalize final embedding
    final_embedding /= np.linalg.norm(final_embedding)
    
    return final_embedding


Generate all book embeddings

In [9]:
book_embeddings = []

for _, row in tqdm(books.iterrows(), total=len(books)):
    emb = create_book_embedding(row, model)
    book_embeddings.append(emb)

book_embeddings = np.vstack(book_embeddings)


100%|██████████| 14610/14610 [10:03<00:00, 24.19it/s]


Store embeddings

In [10]:
np.save('book_embeddings.npy', book_embeddings)
book_embeddings.shape

(14610, 384)

# User Embedding construction

In [283]:
model_user = SentenceTransformer('all-MiniLM-L6-v2')

User definition

In [284]:
user_profile = {
    "favorite_genres": [
        "Mystery & Detective",
        "Suspense",
        "Thrillers"
    ],
    "favorite_subjects": [
        "City Life",
        "Espionage",
        "Social Themes"
    ],
    "preferred_year_range": (2000, 2025),
    "age": 35
}

# Example: (book_index, rating)
user_read_history = [
    (160, 5.0),
    (12036, 4.5)
]

Build User Preferences Text

In [285]:
def build_user_preference_text(user_profile):
    """
    Convert user preferences into natural language text.
    """
    genres = ", ".join(user_profile["favorite_genres"])
    subjects = ", ".join(user_profile["favorite_subjects"])
    start, end = user_profile["preferred_year_range"]
    
    text = (
        f"The reader enjoys {genres}. "
        f"They are interested in {subjects}. "
        f"They prefer books published between {start} and {end}. "
    )
    
    return text

user_pref_text = build_user_preference_text(user_profile)
user_pref_text


'The reader enjoys Mystery & Detective, Suspense, Thrillers. They are interested in City Life, Espionage, Social Themes. They prefer books published between 2000 and 2025. '

Embedd explicit preferences

In [286]:
user_pref_embedding = model_user.encode(
    user_pref_text,
    normalize_embeddings=True
)


Create Reading History Embedding

In [287]:
def build_user_history_embedding(user_read_history, book_embeddings):
    """
    Weighted average of previously liked book embeddings.
    """
    weighted_sum = np.zeros(book_embeddings.shape[1])
    total_weight = 0.0
    
    for book_idx, rating in user_read_history:
        weighted_sum += rating * book_embeddings[book_idx]
        total_weight += rating
        
    if total_weight == 0:
        return None
        
    history_embedding = weighted_sum / total_weight
    history_embedding /= np.linalg.norm(history_embedding)
    
    return history_embedding

user_history_embedding = build_user_history_embedding(
    user_read_history,
    book_embeddings
)

Weights

In [288]:
USER_WEIGHTS = {
    "history": 0.6 if user_history_embedding is not None else 0.0,
    "preferences": 0.4 if user_history_embedding is not None else 1.0
}
print(USER_WEIGHTS)

{'history': 0.6, 'preferences': 0.4}


Final User Embedding

In [289]:
def create_user_embedding(user_pref_embedding, user_history_embedding):
    """
    Combine user preference and history embeddings.
    """
    if user_history_embedding is None:
        final_embedding = user_pref_embedding
    else:
        final_embedding = (
            USER_WEIGHTS["history"] * user_history_embedding +
            USER_WEIGHTS["preferences"] * user_pref_embedding
        )
        
    final_embedding /= np.linalg.norm(final_embedding)
    return final_embedding

user_embedding = create_user_embedding(
    user_pref_embedding,
    user_history_embedding
)

In [290]:
user_embedding.shape

(384,)

In [291]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity with all books
scores = cosine_similarity(
    user_embedding.reshape(1, -1),
    book_embeddings
).flatten()

# Top 10 recommendations
top_k = scores.argsort()[-10:][::-1]
display(scores)
top_k

array([0.58380357, 0.49880896, 0.64361501, ..., 0.46183585, 0.37930949,
       0.68415189])

array([  277, 12036,   160, 10011,  3759, 12080,  5002,  8656,  6733,
        8967])

In [292]:
#Show boocks based on index
df = books.iloc[top_k]
df.head(10)

,ISBN,Title,Author,Year,Publisher,synopsis,subjects,genres,language,pages,binding,image_url,overview
277,0425181111,Strangers,Dean R. Koontz,2002,Berkley Publishing Group,"<b>“The plot twists ingeniously...an engaging,...","Fiction, Thrillers, Suspense, Psychological, S...","Fiction, Thrillers, Suspense",en,704.0,Mass Market Paperback,https://images.isbndb.com/covers/9948383482339...,NaN
12036,0515136824,Wild Rain,Christine Feehan,2004,Jove Books,<b>#1 <i>New York Times</i> bestselling author...,"Fiction, Romance, Paranormal, Shifters, Suspen...","Fiction, Romance, Paranormal",en,384.0,Mass Market Paperback,https://images.isbndb.com/covers/1076012348237...,NaN
160,044023722X,A Painted House,John Grisham,2001,Dell Publishing Company,"""The hill people and the Mexicans arrived on t...","Fiction, Historical, Thrillers, Suspense","Fiction, Historical, Thrillers",en,465.0,Paperback,https://images.isbndb.com/covers/2008052348234...,NaN
10011,0786014199,Paint It Black,P. J. Parrish,2002,Pinnacle Books,<b>Paint It Black</b><br> <br> Florida's Gulf ...,"Fiction, Literary, Thrillers, Suspense","Fiction, Literary, Thrillers",en,411.0,Paperback,https://images.isbndb.com/covers/2329801348246...,NaN
3759,038550120X,A Painted House,JOHN GRISHAM,2001,Doubleday,<b>#1 <i>NEW YORK TIMES</i> BESTSELLER • In a ...,"Fiction, Thrillers, Suspense, Literary, Coming...","Fiction, Thrillers, Suspense",en,400.0,Hardcover,https://images.isbndb.com/covers/6350253482325...,NaN
12080,055380250X,"The Taking (Koontz, Dean)",Dean Koontz,2004,Bantam,In one of the most dazzling books of his celeb...,"FICTION_THRILLERS_SUSPENSE, CALIFORNIA_FICTION","FICTION_THRILLERS_SUSPENSE, CALIFORNIA_FICTION",en,352.0,Hardcover,https://images.isbndb.com/covers/4215893482385...,NaN
5002,0425161633,Second Nature,Alice Hoffman,1998,Berkley Publishing Group,<b>A suburban woman discovers her own wild spi...,"Fiction, Thrillers, Suspense, Women, Family Life","Fiction, Thrillers, Suspense",en,272.0,Paperback,https://images.isbndb.com/covers/9753493482339...,NaN
8656,0804112975,Dangerous Attachments,Sarah Lovett,1996,Ivy Books,"""[A] SUSPENSE-SOAKED THRILLER . . . A welcome ...","Fiction, Mystery & Detective, Police Procedura...","Fiction, Mystery & Detective, Police Procedural",en,344.0,Mass Market Paperback,https://images.isbndb.com/covers/7685243482474...,NaN
6733,0743453468,Whispers at Midnight,Karen Robards,2003,Atria Books,00<br/>From New York Times bestselling author ...,FICTION_THRILLERS_SUSPENSE,FICTION_THRILLERS_SUSPENSE,en,400.0,Hardcover,https://images.isbndb.com/covers/1897754348245...,NaN
8967,0786014202,Thicker Than Water,P. J. Parrish,2003,Pinnacle Books,<b>The Wait Is Over<br>Louis Kincaid Is Back<b...,"Fiction, Literary, Mystery & Detective","Fiction, Literary, Mystery & Detective",en,380.0,Mass Market Paperback,https://images.isbndb.com/covers/2329807348246...,NaN


# Candidate Generation using FAISS and cosine similarity

In [293]:
# %pip install faiss-cpu
import faiss

Verify if embeddings are normalized

In [294]:
def is_normalized(vectors, atol=1e-3):
    norms = np.linalg.norm(vectors, axis=1)
    return np.allclose(norms, 1.0, atol=atol)

print("Book embeddings normalized:", is_normalized(book_embeddings))
print("User embedding normalized:", np.isclose(np.linalg.norm(user_embedding), 1.0))

Book embeddings normalized: True
User embedding normalized: True


Building Index

In [295]:
embedding_dim = book_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

Add Book Embeddings to Index

In [296]:
index.add(book_embeddings)

Search for Nearest Neighbors

In [297]:
TOP_K = 200
scores, indices = index.search(user_embedding.reshape(1, -1), TOP_K)

Display Search Results

In [298]:
candidates = books.iloc[indices[0]].copy()
candidates["similarity_score"] = scores[0]
display(scores[0])

candidates[[
    "Title",
    "Author",
    "Year",
    "genres",
    "subjects",
    "similarity_score"
]].head()

array([0.8014106 , 0.78691494, 0.77823037, 0.77038026, 0.7687419 ,
       0.75805116, 0.75709367, 0.7522385 , 0.7520079 , 0.75192404,
       0.7514684 , 0.75055885, 0.7502283 , 0.7485096 , 0.748097  ,
       0.74514943, 0.74509406, 0.7437227 , 0.7421617 , 0.7419492 ,
       0.7418027 , 0.7415972 , 0.7413865 , 0.73915577, 0.7388618 ,
       0.73812044, 0.73731315, 0.73663455, 0.7365077 , 0.73616844,
       0.73546684, 0.7353    , 0.7351423 , 0.7349981 , 0.7346637 ,
       0.73423827, 0.7341872 , 0.73400486, 0.7314534 , 0.7312286 ,
       0.7300672 , 0.73002553, 0.72907037, 0.72896564, 0.7280474 ,
       0.7279566 , 0.72789174, 0.72779906, 0.727481  , 0.727256  ,
       0.7269584 , 0.7268499 , 0.72674567, 0.7265116 , 0.72650766,
       0.7261045 , 0.7259033 , 0.7256906 , 0.72565484, 0.72564733,
       0.72521716, 0.72516453, 0.724934  , 0.7247832 , 0.7246151 ,
       0.72451305, 0.7244394 , 0.7243848 , 0.7242976 , 0.72405887,
       0.72403854, 0.7240278 , 0.72394663, 0.7233538 , 0.72311

,Title,Author,Year,genres,subjects,similarity_score
277,Strangers,Dean R. Koontz,2002,"Fiction, Thrillers, Suspense","Fiction, Thrillers, Suspense, Psychological, S...",0.801411
12036,Wild Rain,Christine Feehan,2004,"Fiction, Romance, Paranormal","Fiction, Romance, Paranormal, Shifters, Suspen...",0.786915
160,A Painted House,John Grisham,2001,"Fiction, Historical, Thrillers","Fiction, Historical, Thrillers, Suspense",0.778230
10011,Paint It Black,P. J. Parrish,2002,"Fiction, Literary, Thrillers","Fiction, Literary, Thrillers, Suspense",0.770380
3759,A Painted House,JOHN GRISHAM,2001,"Fiction, Thrillers, Suspense","Fiction, Thrillers, Suspense, Literary, Coming...",0.768742
